In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import medicalDataLoader
from progressBar import printProgressBar
from utils_Enes import to_var, getTargetSegmentation, validate, show_predictions
from UNet_Base_Enes import UNet

In [ ]:
# Réglages de l'expérience
MODEL_NAME = 'baseline'   # dossier où le meilleur modèle est enregistré : models/<MODEL_NAME>/
EPOCHS = 50
BATCH_SIZE = 8
LR = 0.001
AUGMENT = False
SEED = 0

# Dossier des données : Kaggle si on y est, sinon le dossier Data à côté du dépôt
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../Data/'

In [ ]:
def runTraining():
    # Même graine à chaque lancement : deux expériences partent du même point
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print('Dataset :', DATA_DIR)
    transform = transforms.Compose([transforms.ToTensor()])

    # train_set sert à apprendre (avec ou sans augmentation) ;
    # train_eval_set contient les mêmes images, jamais modifiées, pour mesurer le Dice d'entraînement
    train_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                      mask_transform=transform, augment=AUGMENT)
    train_eval_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                           mask_transform=transform)
    val_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform,
                                                    mask_transform=transform)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    train_eval_loader = DataLoader(train_eval_set, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    net = UNet(num_classes=4)
    if torch.cuda.is_available():
        net.cuda()
    print('Paramètres : {:,}'.format(sum(p.numel() for p in net.parameters())))

    CE_loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)

    model_dir = os.path.join('models', MODEL_NAME)
    os.makedirs(model_dir, exist_ok=True)
    best_path = os.path.join(model_dir, 'best.pth')
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'val_hd': []}
    best_dsc, best_epoch = -1, 0

    for epoch in range(EPOCHS):
        # Entraînement
        net.train()
        losses = []
        for j, (images, labels, _) in enumerate(train_loader):
            images = to_var(images)
            target = getTargetSegmentation(to_var(labels))

            optimizer.zero_grad()
            loss = CE_loss(net(images), target)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            printProgressBar(j + 1, len(train_loader), prefix='[Training] Epoch: {} '.format(epoch),
                             length=15, suffix=' Loss: {:.4f}'.format(loss.item()))
        train_loss = np.mean(losses)

        # Mesures : Dice sur le train (sans Hausdorff, trop lent), Dice et Hausdorff sur la validation
        _, train_dsc, _ = validate(net, train_eval_loader, CE_loss, with_hausdorff=False)
        val_loss, val_dsc, val_hd = validate(net, val_loader, CE_loss)
        mean_dsc = np.mean(val_dsc)
        print('Epoch {:2d} | loss train {:.4f}  val {:.4f}'.format(epoch, train_loss, val_loss))
        print('   Train Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*train_dsc, np.mean(train_dsc)))
        print('   Val   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*val_dsc, mean_dsc))
        print('   Val   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*val_hd))

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dsc)
        history['val_dice'].append(val_dsc)
        history['val_hd'].append(val_hd)

        # On garde seulement le meilleur modèle sur la validation
        if mean_dsc > best_dsc:
            best_dsc, best_epoch = mean_dsc, epoch
            torch.save(net.state_dict(), best_path)
            print('   -> meilleur modèle enregistré')

    np.save(os.path.join(model_dir, 'history.npy'), history)
    print('Meilleur Dice moyen : {:.4f} (époque {})'.format(best_dsc, best_epoch))

    net.load_state_dict(torch.load(best_path))   # le meilleur, pas le dernier
    return net, train_eval_loader, val_loader

In [ ]:
net, train_loader, val_loader = runTraining()
show_predictions(net, train_loader, title='Entraînement')
show_predictions(net, val_loader, title='Validation')